Feature_Engineering


In [1]:
import pandas as pd

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv ('Cleaned_data.csv')

In [3]:
#df.date = pd.to_datetime(df.date,format="%d/%m/%Y")
#df.set_index('date', inplace=True)
#df.head()

######################################## DAILY PRECENT CHANGE ######################################

In [4]:
df['Pct_Change'] = df.groupby("Name")['close'].pct_change() * 100


#################################### ABS PCT CHANGE ##############################

In [5]:
# יצירת פיצ'ר של עוצמת תנועה יומית (בלי קשר לכיוון)
df['Abs_Pct_Change'] = df['Pct_Change'].abs()


############################## Gap PCT ##################################################

In [6]:
# אנחנו מבקשים מפייתון לעשות את ה-shift בנפרד לכל מניה
df['gap'] = abs((df['open'] - df['close'].groupby(df['Name']).shift(1)) / df['close'].groupby(df['Name']).shift(1)) * 100


########################################חישוב הזנב העליון (תמיד חיובי) ########################################

In [7]:
# 1. חישוב הזנב העליון (תמיד חיובי)
df['Upper_Shadow'] = df['high'] - df[['open', 'close']].max(axis=1)



####################################### חישוב הזנב התחתון (תמיד חיובי)########################################

In [8]:
# 2. חישוב הזנב התחתון (תמיד חיובי)
df['Lower_Shadow'] = df[['open', 'close']].min(axis=1) - df['low']


########################## סך הכל זנבות בדולרים #######################

In [9]:
# 3. סך הכל זנבות בדולרים
df['Total_Shadow_USD'] = df['Upper_Shadow'] + df['Lower_Shadow']


##################################### סך הכל זנבות כאחוז מהטווח היומי ################################

In [10]:
# 4. סך הכל זנבות כאחוז מהטווח היומי (זה הפיצ'ר הכי חזק לקלסטרינג!)
df['Shadow_to_Range'] = df['Total_Shadow_USD'] / (df['high'] - df['low'])


In [11]:
##################### תנודתיות תוך-יומית (High-Low Range) #####################

df['Daily_Range'] = ((df['high'] - df['low']) / df['close']) * 100

################################ התשואה היומית של המניה באחוזים ###############################

In [12]:
# מחושב בדרך כלל ברמה השבועית או החודשית, אבל כאן נכין את הבסיס היומי
df['Daily_Return'] = df.groupby('Name')['close'].pct_change()


############################## STD ###############################

In [13]:
df['Rolling_Volatility_10D'] = (
    df.groupby('Name')['Daily_Return']
    .rolling(window=10)
    .std()
    .reset_index(level=0, drop=True)
)


################################ שינוי בנפח המסחר (Volume Change) ############################

In [14]:
df['Volume_Pct_Change'] = df.groupby('Name')['volume'].pct_change() * 100


######################################### VOLUME STD ###################################

In [15]:
# נחשב את סטיית התקן של השינוי בווליום - זה מראה כמה הווליום "קופצני"
df['Vol_Instability'] = df.groupby('Name')['Volume_Pct_Change'].rolling(window=10).std().reset_index(level=0, drop=True)



################################### מדד מחיר מול ממוצע נע (Distance from MA) #######################################

In [16]:
df['MA20'] = df.groupby('Name')['close'].transform(lambda x: x.rolling(window=20).mean())
df['Dist_From_MA20'] = ((df['close'] - df['MA20']) / df['MA20']) * 100


In [17]:
# שמירת הדאטה הנקי לקובץ CSV חדש
# index=True חשוב כאן כי הגדרנו את התאריך כאינדקס ואנחנו רוצים שהוא יישמר בקובץ
df.to_csv('Feature_Engineering_Data.csv', index=True)

print("הקובץ Feature_Engineering_Data.csv נשמר בהצלחה!")

הקובץ Feature_Engineering_Data.csv נשמר בהצלחה!
